<details>
<summary>How to read this notebook</summary>

<br>

📌 *Every note marked with this pin was added by me — they are not part of the original notebook.*

Each pin points at something in the cell just **above or below** it and asks you a question.
Stop for a few seconds and **answer it in your head** before moving on.

When there is an `answer — click to reveal` block, open it **only after** you have formed your own opinion.

> 💡 The code itself is intentionally written the way real notebooks are written.
> It runs, and the results are fine. The problem is what happens when **someone else inherits it**.

</details>

---

# Gym Churn - model v2 final FINAL

predicting churn for the retention team

---

📌 *you have been given this notebook by a colleague who just left for holiday.*
*he says the model is good and ready to ship.*
*your job today is to read it, run it, spot the problems, and eventually rewrite it properly.*

*start by fixing the paths below so the notebook runs end to end.*

In [ ]:
import numpy as np
import pandas as pd

## 1. load data

📌 **your turn, the notebook does not run yet.**

*your data is in the* `data/01_raw/` *folder of this repository:*
`members.csv` *and* `checkins.parquet`*.*
*edit the two paths so the cell runs, then continue.*

*try it yourself first — the answer is right after the cell if you get stuck.*

In [ ]:
# TODO: fix these paths before running
members = pd.read_csv(
    "/Workspace/Users/alban.thuet@axa.com/some/old/folder/members_final_v2.csv"
)
checkins = pd.read_parquet(
    "/Workspace/Users/alban.thuet@axa.com/another/folder/checkins_export_jan.parquet"
)

print(members.shape)
print(checkins.shape)

<details>
<summary>answer — click to reveal</summary>

the notebook lives in `notebooks/`, so the data is one folder up:

```python
members  = pd.read_csv("../data/01_raw/members.csv")
checkins = pd.read_parquet("../data/01_raw/checkins.parquet")
```

you should get `(8640, 12)` for members and `(150062, 5)` for checkins.

</details>

---
📌 *you just edited two paths to make someone else's notebook run on your machine.*
*now imagine the data moves to a new folder next month, and this notebook has fifteen read/write calls*
*spread over eighty cells. how do you find them all? how do you know you found them all?*

In [ ]:
members.head()

In [ ]:
members.isna().sum()

In [ ]:
members.describe()

In [ ]:
checkins.head()

In [ ]:
checkins["activity"].value_counts()

In [ ]:
checkins["days_ago"].describe()

## 2. target

a member is **churned** if they have no check-in in the last 30 days.

In [ ]:
recent = checkins[checkins["days_ago"] <= 30]
active_ids = recent["member_id"].unique()

members["churned"] = (~members["member_id"].isin(active_ids)).astype(int)

print(members["churned"].value_counts())
print(f"churn rate: {members['churned'].mean():.1%}")

📌 *the number `30` is the business definition of churn. it is buried inside a filter with no label.*
*what happens when the retention team says "let's use 45 days instead"?*

<details>
<summary>answer — click to reveal</summary>

you would have to search every cell manually. `30` also appears again further down in the checkin features.
miss one and your features no longer match your target definition.
this is a parameter — it should be defined once, at the top, with a name.

</details>

## 3. checkin features

In [ ]:
history = checkins[checkins["days_ago"] > 30]

checkin_features = (
    history.groupby("member_id")
    .agg(
        past_checkins=("checkin_id", "count"),
        past_avg_duration=("duration_minutes", "mean"),
        past_n_activities=("activity", "nunique"),
    )
    .reset_index()
)

members = members.merge(checkin_features, on="member_id", how="left")

fill_cols = ["past_checkins", "past_avg_duration", "past_n_activities"]
members[fill_cols] = members[fill_cols].fillna(0)

members.head()

📌 *this block does two things at once — it aggregates, then it merges.*
*if the aggregation logic changes, how do you test just that part without rerunning everything above?*

<details>
<summary>answer — click to reveal</summary>

you cannot. you have to rerun the whole notebook from the top.
wrapping this in a function like `build_checkin_features(members, checkins)` means
you can call it independently, test it, and change it without touching anything else.

</details>

## 4. cleaning

In [ ]:
print(f"before: {len(members)}")
members = members.drop_duplicates()
print(f"after:  {len(members)}")

In [ ]:
members["age"] = members["age"].fillna(members["age"].median())
members["distance_km"] = members["distance_km"].fillna(members["distance_km"].median())
members["contract_type"] = members["contract_type"].fillna("Monthly")

members["club"] = members["club"].str.strip().str.title()
members["signup_channel"] = members["signup_channel"].str.strip()

members.isna().sum()

📌 *`"Monthly"` in line 3 is a business decision — if a member has no contract type, we assume Monthly.*
*nothing in the code signals that this was intentional.*
*how would a new team member know this was a choice and not just a random default?*

<details>
<summary>answer — click to reveal</summary>

they would not. they might change it, delete it, or copy it wrong.
fill values like this belong in a config — named, visible, and easy to find.

</details>

In [ ]:
import matplotlib.pyplot as plt  # importing here, right where the chart is

members["age"].hist(bins=30)
plt.title("age distribution")
plt.show()

📌 *`matplotlib` is imported here, in the middle of the notebook, because this is the first chart.*
*the same thing happens again in sections 5, 6, 7, 8 and 10 — each new library appears*
*the moment it is first needed. it feels natural while you write. what does it cost the next reader?*

<details>
<summary>answer — click to reveal</summary>

**you cannot tell what this notebook depends on without reading every single cell.**
there is no one place that answers "what do I need to install to run this?".

it also breaks partial execution. say a colleague only wants to redraw the ROC curve
in section 8. they open a fresh kernel, run section 8, and get:

```
NameError: name 'plt' is not defined
```

because `plt` was imported back in section 4, in a cell that has nothing to do with plotting.
now they have to hunt upwards through the notebook to find which cells they are secretly
depending on — and that dependency is written nowhere.

imports belong at the top: one block, visible immediately, independent of execution order.

</details>

In [ ]:
members["club"].value_counts()

In [ ]:
members["contract_type"].value_counts()

## 5. encode

In [ ]:
from sklearn.preprocessing import LabelEncoder  # another import mid-notebook

cat_cols = ["contract_type", "club", "signup_channel"]

le = LabelEncoder()
for col in cat_cols:
    members[col] = le.fit_transform(members[col].astype(str))

members[cat_cols].head()

📌 *`cat_cols` is defined here. the same column names appear again in `feature_cols` further down.*
*if you add a new categorical column, will you remember to update both places?*

<details>
<summary>answer — click to reveal</summary>

probably not. miss one and the model silently trains without the new column.
a single `CAT_COLS = [...]` defined once at the top removes this risk entirely.

</details>

## 6. split + save test set

In [ ]:
from sklearn.model_selection import train_test_split  # another import mid-notebook

feature_cols = [
    "age",
    "tenure_months",
    "monthly_fee_eur",
    "distance_km",
    "baseline_visits_pm",
    "has_personal_trainer",
    "joined_with_friend",
    "group_classes_pm",
    "contract_type",
    "club",
    "signup_channel",
    "past_checkins",
    "past_avg_duration",
    "past_n_activities",
]

X = members[feature_cols]
y = members["churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# save test for evaluation later
test_set = X_test.copy()
test_set["churned"] = y_test.values
test_set.to_csv(
    "../data/01_raw/test_set_DO_NOT_DELETE.csv",
    index=False,
)

print(f"train: {X_train.shape}")
print(f"test:  {X_test.shape}")
print(f"churn rate train: {y_train.mean():.1%}")
print(f"churn rate test:  {y_test.mean():.1%}")

📌 *the test set is saved here, inside the training section.*
*what happens if someone runs the evaluation cells without running this one first?*

<details>
<summary>answer — click to reveal</summary>

they silently evaluate on stale data from a previous run — or worse, on the wrong model entirely.
the dependency between the two sections is completely invisible.
this is one of the most common sources of silent errors in notebook-based workflows.

</details>

## 7. train

In [ ]:
from sklearn.ensemble import RandomForestClassifier  # another import mid-notebook

# tried 100 trees first, 200 seems better
model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)

model.fit(X_train, y_train)
print("done")

📌 *`n_estimators=200`, `max_depth=10`, `random_state=42` — all hardcoded.*
*are these the same values that produced the model currently in production?*

<details>
<summary>answer — click to reveal</summary>

there is no way to know. to run an experiment with different values you edit the code directly.
to reproduce a previous run you have to remember what you typed.
these are parameters — they belong in a config, not scattered in cells.

</details>

## 8. results

In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve

y_proba = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

auc = roc_auc_score(y_test, y_proba)
print(f"AUC: {auc:.4f}")

In [ ]:
cm = confusion_matrix(y_test, y_pred, normalize="true")

plt.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.colorbar()

classes = ["Active", "Churned"]
tick_marks = np.arange(len(classes))
plt.xticks(tick_marks, classes, rotation=45)
plt.yticks(tick_marks, classes)

thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(
            j,
            i,
            str(np.round(cm[i, j], 2)),
            horizontalalignment="center",
            color="white" if cm[i, j] > thresh else "black",
        )

plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.tight_layout()
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)

plt.plot(fpr, tpr, label=f"AUC = {auc:.2f}")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.show()

## 9. feature importance

In [ ]:
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(
    ascending=False
)
print(importances)

In [ ]:
importances.plot(kind="bar", figsize=(13, 5), title="Feature Importance")
plt.ylabel("importance")
plt.tight_layout()
plt.show()

📌 *feature importance is useful — but in notebook form, this output is easy to lose.*
*where is this chart saved? how do you compare it to last week's model run?*

<details>
<summary>answer — click to reveal</summary>

nowhere by default. it appears on screen, then disappears.
if you want traceability, charts and metrics must be written to named outputs,
not just shown in a transient notebook session.

</details>

## 10. save model

In [ ]:
import pickle  # another import mid-notebook

# v2 - updated after retrain
with open("churn_model_v2_final.pkl", "wb") as f:
    pickle.dump(model, f)

print("saved")

📌 *every time this cell runs, `churn_model_v2_final.pkl` is overwritten.*
*which model is currently running in production? when was it trained? with what parameters?*

<details>
<summary>answer — click to reveal</summary>

nobody knows. there is no versioning, no timestamp, no record of the parameters used.
if you retrain tomorrow with different values, today's model is gone forever.

</details>